In [1]:
import scanpy as sc
import pandas as pd


/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
old_pd = sc.read_mtx('/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/knoblich/GSM7855409_232029_matrix.mtx').T

In [3]:
old_pd.X.max()

3678.0

In [4]:
gene_names = pd.read_table("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/knoblich/GSM7855409_232029_features.tsv", header=None)[1]

In [5]:
gene_names

0        MIR1302-2HG
1            FAM138A
2              OR4F5
3         AL627309.1
4         AL627309.3
            ...     
36596     AC141272.1
36597     AC023491.2
36598     AC007325.1
36599     AC007325.4
36600     AC007325.2
Name: 1, Length: 36601, dtype: object

In [6]:
cell_names = pd.read_table("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/knoblich/GSM7855409_232029_barcodes.tsv", header=None)[0]

In [7]:
cell_names

0        AAACCCAAGCCTCATA-1
1        AAACCCAAGGATACAT-1
2        AAACCCAAGTCGCCAC-1
3        AAACCCACAATCGCAT-1
4        AAACCCACAGGAGGAG-1
                ...        
17127    TTTGTTGTCGGACCAC-1
17128    TTTGTTGTCGTTACCC-1
17129    TTTGTTGTCTGATGGT-1
17130    TTTGTTGTCTTCGCTG-1
17131    TTTGTTGTCTTCTCAA-1
Name: 0, Length: 17132, dtype: object

In [8]:
lineage_barcodes = pd.read_csv("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/knoblich/GSM7855413_232029_lineage_barcodes.csv")
                                                                                                                                                        

In [9]:
lineage_barcodes.set_index("cellbarcode", inplace=True)

In [10]:
lineage_barcodes["lid"].unique().shape

(200,)

In [11]:
old_pd.obs_names = cell_names

In [12]:
old_pd.var_names = gene_names

In [13]:
old_pd.obs["barcode"] = old_pd.obs_names.map(lineage_barcodes["lid"])

In [14]:
adata_no_nan = old_pd[old_pd.obs["barcode"].notna()]

In [15]:

# Mark MT genes
adata_no_nan.var["mt"] = adata_no_nan.var_names.str.upper().str.startswith("MT-")

# Compute MT % per cell
sc.pp.calculate_qc_metrics(adata_no_nan, qc_vars=["mt"], inplace=True)

# Filter cells with >3.5% MT
adata_no_nan = adata_no_nan[adata_no_nan.obs["pct_counts_mt"] <= 3.5].copy()


/tmp/ipykernel_2597836/476436537.py:2: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata_no_nan.var["mt"] = adata_no_nan.var_names.str.upper().str.startswith("MT-")
/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [16]:
sc.pp.filter_cells(adata_no_nan, min_genes=100)
sc.pp.filter_genes(adata_no_nan, min_cells=2)

/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [17]:
# preprocess
sc.pp.normalize_total(adata_no_nan, target_sum=1e4)
#sc.pp.log1p(adata_no_nan)
# highly variable genes
#sc.pp.highly_variable_genes(adata_no_nan, n_top_genes=N_HVG, flavor="cell_ranger")


In [18]:

adata_filtered = adata_no_nan#[:,  adata_no_nan.var.highly_variable]

In [19]:
df = pd.DataFrame(adata_filtered.X.todense(), columns = adata_filtered.var_names)

In [20]:
df

,MIR1302-2HG,AL627309.1,AL627309.5,AL627309.4,AP006222.2,AL669831.2,LINC01409,FAM87B,LINC01128,LINC00115,...,MT-ND5,MT-ND6,MT-CYB,AC011043.1,AC011043.2,AC011841.1,AC240274.1,AC004556.3,AC007325.4,AC007325.2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,5.693680,0.948947,2.846840,0.948947,0.0,0.0,0.000000,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,5.365692,3.714710,26.828463,1.238237,0.0,0.0,0.000000,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.6072,0.0,...,9.643202,0.000000,12.857602,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,2.091613,1.568709,4.183225,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8725,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,4.786161,3.076818,27.691360,0.341869,0.0,0.0,0.341869,0.000000,0.0,0.0
8726,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,3.324284,1.108095,13.297134,0.554047,0.0,0.0,0.000000,0.000000,0.0,0.0
8727,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,11.792453,0.000000,29.481133,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
8728,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,4.706325,0.000000,37.650600,0.941265,0.0,0.0,0.000000,0.000000,0.0,0.0


In [21]:
df

,MIR1302-2HG,AL627309.1,AL627309.5,AL627309.4,AP006222.2,AL669831.2,LINC01409,FAM87B,LINC01128,LINC00115,...,MT-ND5,MT-ND6,MT-CYB,AC011043.1,AC011043.2,AC011841.1,AC240274.1,AC004556.3,AC007325.4,AC007325.2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,5.693680,0.948947,2.846840,0.948947,0.0,0.0,0.000000,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,5.365692,3.714710,26.828463,1.238237,0.0,0.0,0.000000,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.6072,0.0,...,9.643202,0.000000,12.857602,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,2.091613,1.568709,4.183225,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8725,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,4.786161,3.076818,27.691360,0.341869,0.0,0.0,0.341869,0.000000,0.0,0.0
8726,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,3.324284,1.108095,13.297134,0.554047,0.0,0.0,0.000000,0.000000,0.0,0.0
8727,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,11.792453,0.000000,29.481133,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0
8728,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0000,0.0,...,4.706325,0.000000,37.650600,0.941265,0.0,0.0,0.000000,0.000000,0.0,0.0


In [22]:
df["barcode"] = adata_filtered.obs["barcode"].values

In [23]:
barcode_df = {}
k = 0
for i in df["barcode"].unique():
    barcode_df[i] = k
    k +=1

In [24]:
df["barcode_cont"] = df["barcode"].map(barcode_df)

In [25]:
df["barcode_cont"].unique().shape == df["barcode"].unique().shape

True

In [26]:
df.drop(columns = ["barcode"], inplace = True)

In [27]:
df.to_pickle("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered_nolog/knoblich_day16_2_nolog.pkl")

